# Soil Classification — Ziegenrück Catchment

**Purpose:** Classifies soil horizons from the Thuringia state soil database
(BODENREDU.xlsx) into the three texture groups required by TALSIM-NG
(Sand / Loam-silt / Clay) and converts raw horizon data into model-ready units.

**What it does:**
- Reads the soil raster class numbers present in the watershed (KLASSE input)
- Maps KA4 texture codes → Sand (1) / Loam-silt (2) / Clay (3)
- Converts pF values and depth to field capacity, wilting point, and
  saturation in mm/m; converts Kf from cm/day → mm/h
- Exports a wide-format table per horizon (A–D) to CSV

**Input:** `BODENREDU.xlsx` (Thuringia soil database)  
**Output:** `Soil-classification by Klasse_Zigenrueck.csv`

---

In [ ]:
import pandas as pd
import os

### File Path

In [ ]:
file_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\Boden und Landnutzung\UTM32\BODENREDU.xlsx"
df = pd.read_excel(file_path)


### Output Path

In [ ]:
output_folder = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\Boden und Landnutzung"
os.makedirs(output_folder, exist_ok=True)
output_file_path = os.path.join(output_folder, "Soil-classification-wide_ZR.xlsx")

### Soil Category 
Based on the standard taxonomic soil classification of Germany

In [ ]:
def soil_category(ka4_code):
    if pd.isna(ka4_code):
        return "Unknown"
    if ka4_code.startswith(("Ss", "Sl2", "Su2", "Sl3", "St2", "Su3", "Su4", "fS", "mS", "gS")):
        return "Sand"
    elif ka4_code.startswith(("Slu", "Sl4", "St3", "Ls2", "Ls3", "Ls4", "Lt2", "Lts", "Ts4", "Ts3", "Uu", "Us", "Ut2", "Ut3", "Uls","Ut4", "Lu", "Lt3")):
        return "Loam/silt"
    elif ka4_code.startswith(("Tu3", "Tu4", "Ts2", "Tl", "Tu2", "Tt")):
        return "Clay"
    return "Unknown"


def classify_bd_class(ka4_code):
    soil_cat = soil_category(ka4_code)
    if soil_cat == "Sand":
        return 1
    elif soil_cat == "Loam/silt":
        return 2
    elif soil_cat == "Clay":
        return 3
    else:
        return 0  # Unknown or unclassified

### User Defined Soil Class Number
Based on the clipped soil raster file from whole Thüringia

In [ ]:

klasse_input = input("Enter one or more KLASSE numbers (comma-separated): ").strip()
klasse_list = [k.strip() for k in klasse_input.split(",")]

# Ziegnerück EZG*Soil raster has these classes 12,23,31,32,51,62,64,75,92

### Paired with S-Value
This is optional. In our case we already used the processed data that has known S-value following each class.

In [ ]:
klasse_svalue_pairs = []

while True:
    klasse = input("Enter KLASSE number (press Enter to start analysis): ").strip()
    if not klasse:  # User pressed Enter → break loop
        break

    s_value_input = input(f"Enter one or more S_VALUEs for KLASSE {klasse} (comma-separated, or type 'all'): ").strip()

    if s_value_input.lower() == 'all':
        # Add all S_VALUEs for this KLASSE
        df_matches = df[df["KLASSE"].astype(str) == klasse]
        if df_matches.empty:
            print(f"⚠️  No data found for KLASSE = {klasse}")
        else:
            for sv in df_matches["S_VALUE"].astype(str).unique():
                klasse_svalue_pairs.append((klasse, sv))
    else:
        s_values = [sv.strip() for sv in s_value_input.split(",") if sv.strip()]
        for sv in s_values:
            klasse_svalue_pairs.append((klasse, sv))
#12,23,31,32,51,62,64,75,92

### Analysis
This function will convert the absolute data for soil corresponding to each horizon to the required format and unit by TALSIM-NG

In [ ]:
klasse_data = {}

for klasse in klasse_list:
    df_klasse = df[df["KLASSE"].astype(str) == klasse]

    if df_klasse.empty:
        print(f"\n⚠️  No data found for KLASSE = {klasse}")
        continue

    row = df_klasse.iloc[0]
    excel_row_number = df_klasse.index[0] + 2
    horizons = ["A", "B", "C", "D"]

    klasse_row = {
        "KLASSE": klasse,
    }

    for h in horizons:
        try:
            depth = row[f"{h}TIEFE"]
            pf18 = row[f"{h}PF18"]
            pf42 = row[f"{h}PF42"]
            pf0 = row[f"{h}PF0"]
            kf_cm_day = row[f"{h}KF"]
            subs = row[f"{h}SUBS"]

            if pd.isna(depth) or depth == 0:
                continue

            # Compute derived values
            depth_m = depth / 100
            #category = soil_category(str(subs))
            bd_class = classify_bd_class(str(subs))

            fc_mm = (pf18 / 100) * depth * 10
            wp_mm = (pf42 / 100) * depth * 10
            sat_mm = (pf0 / 100) * depth * 10

            fc_mm_m = fc_mm / depth_m
            wp_mm_m = wp_mm / depth_m
            sat_mm_m = sat_mm / depth_m

            kf_mm_hr = round((kf_cm_day * 10) / 24, 2) if pd.notna(kf_cm_day) else "N/A"
            max_infiltration = round(kf_mm_hr * 1.01, 2) if isinstance(kf_mm_hr, float) else "N/A"

            # Add to klasse_row
            klasse_row[f"{h}_Soil_Category"] = bd_class
            klasse_row[f"{h}_BD_Class"] = bd_class
            klasse_row[f"{h}_Soil_Texture"] = subs
            klasse_row[f"{h}_WP_mm_m"] = round(wp_mm_m, 2)
            klasse_row[f"{h}_FC_mm_m"] = round(fc_mm_m, 2)
            klasse_row[f"{h}_SAT_mm_m"] = round(sat_mm_m, 2)
            klasse_row[f"{h}_Kf_mm/h"] = kf_mm_hr
            klasse_row[f"{h}_Max_inf_mm/h"] = max_infiltration
            klasse_row[f"{h}_depth_m"] = round(depth_m, 2)
    
        except KeyError as e:
            print(f"⚠️  Missing column: {e}")
            continue
            
    # Use both KLASSE and S_VALUE to uniquely identify this entry
    #klasse_data[f"{klasse}_{s_value}"] = klasse_row
    klasse_data[klasse] = klasse_row

# Export results
result_df = pd.DataFrame(list(klasse_data.values()))
result_df.to_csv("Soil-classification by Klasse_Zigenrueck.csv", index=False)
#result_df.to_excel("Soil-classification-wide.xlsx", index=False)

if  result_df.empty:
    print("\n⚠️ Done")
else:
    print("\n✅ Done")

In [ ]:
if not result_df.empty:
    print("\n✅ Wide-format Processed Data:\n")
    print(result_df)
else:
    print("\n⚠️  No valid data processed.")